# 03 Sample Split and Overfitting

第三课目标：理解样本内、验证集、样本外，以及为什么不能只选历史最优参数。

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from quant_trading.market_data import download_ohlcv
from quant_trading.validation import (
    compare_parameter_across_periods,
    evaluate_moving_average_grid,
    format_parameter_table,
    pivot_metric,
    plot_parameter_heatmaps,
    select_top_parameters,
)

## 1. 划分样本

时间序列不能随机打乱，要按时间切分。

- `train`：2000-2014，用来研究和选候选参数
- `valid`：2015-2019，用来比较少数候选参数
- `test`：2020-2026，只用于最终检验

In [ ]:
SYMBOL = "SPY"
df = download_ohlcv(symbol=SYMBOL, start="2000-01-01", auto_adjust=True)
df.index.min(), df.index.max(), len(df)

## 2. 测试一个小参数网格

这里不是为了找圣杯，而是观察参数在不同时间段是否稳定。

In [ ]:
short_windows = [10, 20, 50]
long_windows = [50, 100, 200]

results = evaluate_moving_average_grid(
    df,
    short_windows=short_windows,
    long_windows=long_windows,
    transaction_cost_bps=1.0,
)

results.head()

## 3. 看训练区间表现最好的参数

这里用 Calmar 比率排序。Calmar 比率 = 年化收益 / 最大回撤绝对值。

In [ ]:
top_train = select_top_parameters(
    results,
    period="train",
    metric="strategy_calmar",
    n=5,
)
top_train[
    [
        "short_window",
        "long_window",
        "strategy_annualized_return",
        "strategy_max_drawdown",
        "strategy_calmar",
        "trades",
        "time_in_market",
    ]
]

## 4. 把训练区最优参数拿到验证集和测试集看

如果训练区很好、验证区和测试区很差，就要警惕过拟合。

In [ ]:
best = top_train.iloc[0]
selected = compare_parameter_across_periods(
    results,
    short_window=int(best["short_window"]),
    long_window=int(best["long_window"]),
)

selected[
    [
        "period",
        "short_window",
        "long_window",
        "strategy_annualized_return",
        "strategy_max_drawdown",
        "strategy_calmar",
        "trades",
        "time_in_market",
    ]
]

## 5. 参数热力图

如果只有某一个参数点特别好，附近都很差，过拟合风险更高。

In [ ]:
pivot_metric(results, period="train", metric="strategy_annualized_return")

In [ ]:
fig = plot_parameter_heatmaps(results, metric="strategy_annualized_return")
fig

## 6. 本课结论

不要相信单一历史最优参数。更可靠的策略，应该在不同时间段、相邻参数、成本压力下都不至于崩。